# Evolving a Lunar Lander with differentiable Genetic Programming

## Installation
To install the required libraries run the command:

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Updated short names tailored for your presentation slide
labels = [
    'Baseline',
    'Landing Close to Center',
    'Low Landing Velocity',
    'Near-Zero Landing Angle',
    'Dampening Horizontal Velocity',
    'Preventing Conflicting Engines'
]
scores = [-602.78, -1242.58, -1415.84, -1393.14, -603.36, -1206.26]

# Set up the figure size to fit long horizontal labels cleanly
plt.figure(figsize=(12, 6))

# Plot bars using a clean blue hue
bars = plt.bar(labels, scores, color='#336699', edgecolor='none', width=0.5)


# Customize axes and grid
plt.ylabel('Fitness Score (Higher is Better)', fontsize=11, fontweight='bold')
plt.title('Lunar Lander Fitness Functions: Strategy Variations vs. Baseline', fontsize=13, fontweight='bold')
plt.xticks(rotation=25, ha='right')  # Rotated for slide legibility

# Add horizontal gridlines behind the bars
plt.gca().set_axisbelow(True)
plt.grid(axis='y', linestyle='--', color='#cccccc')

# Remove the top and right spines for a clean look
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)

# Add text values underneath each bar for precision
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval - 45, f'{yval:.2f}',
             ha='center', va='top', fontweight='bold', color='#333333')

# Tighten layout and render
plt.tight_layout()
plt.show()

In [ ]:
# !pip install -r requirements-pinned.txt

## Imports
Imports from the standard genepro-multi library are done here. Any adjustments (e.g. different operators) should be made in the notebook. For example:

```
class SmoothOperator(Node):
  def __init__(self):
    super(SmoothOperator,self).__init__()
    self.arity = 1
    self.symb = "SmoothOperator"

  def _get_args_repr(self, args):
    return self._get_typical_repr(args,'before')

  def get_output(self, X):
    c_outs = self._get_child_outputs(X)
    return np.smoothOperation(c_outs[0])

  def get_output_pt(self, X):
    c_outs = self._get_child_outputs_pt(X)
    return torch.smoothOperation(c_outs[0])
```

In [ ]:
import gymnasium as gym

from genepro.node_impl import *
from genepro.evo import Evolution
from genepro.node_impl import Constant

import torch
import torch.optim as optim

import random
import os
import copy
from collections import namedtuple, deque

import matplotlib.pyplot as plt
from matplotlib import animation

SEED = 42

np.random.seed(SEED)
random.seed(SEED)

## Reinforcement Learning Setup
Here we first setup the Gymnasium environment. Please see https://gymnasium.farama.org/environments/box2d/lunar_lander/ for more information on the environment. 

Then a memory buffer is made. This is a buffer in which state transitions are stored. When the buffer reaches its maximum capacity old transitions are replaced by new ones.

A frame buffer is initialised used to later store animation frames of the environment.

In [ ]:
env = gym.make("LunarLander-v2", render_mode="rgb_array")

In [ ]:
Transition = namedtuple('Transition', ('state', 'action', 'next_state', 'reward'))

class ReplayMemory(object):
    def __init__(self, capacity):
        self.memory = deque([], maxlen=capacity)

    def push(self, *args):
        """Save a transition"""
        self.memory.append(Transition(*args))

    def sample(self, batch_size):
        return random.sample(self.memory, batch_size)

    def __len__(self):
        return len(self.memory)

    def __iadd__(self, other):
      self.memory += other.memory
      return self 

    def __add__(self, other):
      self.memory = self.memory + other.memory 
      return self

In [ ]:
frames = []

## Fitness Function

Here you get to be creative. The default setup evaluates 5 episodes of 300 frames. Think of what action to pick and what fitness function to use. The Multi-tree takes an input of $n \times d$ where $n$ is a batch of size 1.

In [ ]:
def fitness_function_pt(multitree, num_episodes=5, episode_duration=300, render=False, ignore_done=False):
  memory = ReplayMemory(10000)
  rewards = []

  for ep in range(num_episodes):
    # get initial state of the environment
    observation = env.reset(seed=SEED + ep)
    observation = observation[0]

    for _ in range(episode_duration):
      if render:
        frames.append(env.render())

      input_sample = torch.from_numpy(observation.reshape((1,-1))).float()

      action = torch.argmax(multitree.get_output_pt(input_sample))
      observation, reward, terminated, truncated, info = env.step(action.item())
      rewards.append(reward)
      output_sample = torch.from_numpy(observation.reshape((1,-1))).float()
      memory.push(input_sample, torch.tensor([[action.item()]]), output_sample, torch.tensor([reward]))
      if (terminated or truncated) and not ignore_done:
        break

  fitness = np.sum(rewards)

  return fitness, memory

In [ ]:
CURRENT_EPISODE_DURATION = 150

def fitness_function_pt3(
    multitree,
    num_episodes=10,
    episode_duration=None,
    tree_penalty=0.1
):
    global CURRENT_EPISODE_DURATION

    if episode_duration is None:
        episode_duration = CURRENT_EPISODE_DURATION

    memory = ReplayMemory(10000)
    episode_returns = []

    for ep in range(num_episodes):
        observation = env.reset(seed=SEED + ep)
        observation = observation[0]

        episode_reward = 0

        for _ in range(episode_duration):
            input_sample = torch.from_numpy(
                observation.reshape((1, -1))
            ).float()

            action = torch.argmax(
                multitree.get_output_pt(input_sample)
            )

            observation, reward, terminated, truncated, _ = env.step(action.item())

            x = abs(observation[0])
            y = observation[1]
            vx = abs(observation[2])
            vy = abs(observation[3])
            angle = abs(observation[4])

            reward -= 0.5 * x
            reward -= 0.2 * vx
            reward -= 0.3 * vy
            reward -= 0.2 * angle

            episode_reward += reward

            output_sample = torch.from_numpy(
                observation.reshape((1, -1))
            ).float()

            memory.push(
                input_sample,
                torch.tensor([[action.item()]]),
                output_sample,
                torch.tensor([reward])
            )

            if terminated or truncated:
                break

        episode_returns.append(episode_reward)

    mean_return = np.mean(episode_returns)
    std_return = np.std(episode_returns)

    tree_size = sum(len(child) for child in multitree.children)

    fitness = (
        mean_return
        - 0.25 * std_return
        - tree_penalty * tree_size
    )

    return fitness, memory

## Evolution Setup
Here the leaf and internal nodes are defined. Think about the odds of sampling a constant in this default configurations. Also think about any operators that could be useful and add them here. 

Adjust the population size (multiple of 8 if you want to use the standard tournament selection), max generations and max tree size to taste. Be aware that each of these settings can increase the runtime.

In [ ]:
num_features = env.observation_space.shape[0]
leaf_nodes = [Feature(i) for i in range(num_features)]
leaf_nodes = leaf_nodes + [Constant()] # Think about the probability of sampling a coefficient
# internal_nodes = [Plus(),Minus(),Times(),Div()] #Add your own operators here
internal_nodes = [Plus(), Minus(), Times(), Sqrt(), Log(), Max(), Tanh(), Abs(), Div()] #Add your own operators here

## Evolve
Running this cell will use all the settings above as parameters

In [ ]:
# RUN_CURRICULUM = False
RUN_CURRICULUM = True

if RUN_CURRICULUM:
    import copy
    import time

    CURRENT_EPISODE_DURATION = 150

    evo = Evolution(
        fitness_function_pt3,
        internal_nodes,
        leaf_nodes,
        4,
        pop_size=128,
        max_gens=50,
        max_tree_size=63,
        n_jobs=8,
        verbose=True,
        init_method="ramped_half_and_half",
        elite_archive_size=0,
        seed=SEED
    )

    curriculum = [
        (150, 15),
        (300, 15),
        (500, 20),
    ]

    best_fitness = -np.inf

    evo.start_time = time.time()
    evo._initialize_population()

    for duration, generations in curriculum:
        print(f"\nStage: {duration} steps for {generations} generations")

        CURRENT_EPISODE_DURATION = duration

        for _ in range(generations):
            evo._perform_generation()

            current_best = evo.best_of_gens[-1]

            if current_best.fitness > best_fitness:
                best_fitness = current_best.fitness
                best_tree = copy.deepcopy(current_best)

            print(
                f"gen: {evo.num_gens}, "
                f"duration: {duration}, "
                f"best fitness: {current_best.fitness:.3f}, "
                f"size: {len(current_best)}"
            )
else:
    evo = Evolution(
        fitness_function_pt,
        internal_nodes,
        leaf_nodes,
        4,
        pop_size=128,
        max_gens=50,
        max_tree_size=63,
        n_jobs=16,
        verbose=True,
        init_method="ramped_half_and_half",
        elite_archive_size=0,
        seed=SEED
    )

    evo.evolve()

# Test

In [ ]:
VALIDATION_SEEDS = [10_000 + i for i in range(20)]
INVALID_FITNESS = -1e9

def _make_diagnostics(episode_returns=None, episode_lengths=None, successes=0, crashes=0, timeouts=0, invalid_outputs=0):
  episode_returns = episode_returns or []
  episode_lengths = episode_lengths or []
  returns = np.array(episode_returns, dtype=float)

  return {
    "mean_return": float(np.mean(returns)) if len(returns) else INVALID_FITNESS,
    "std_return": float(np.std(returns)) if len(returns) else 0.0,
    "min_return": float(np.min(returns)) if len(returns) else INVALID_FITNESS,
    "max_return": float(np.max(returns)) if len(returns) else INVALID_FITNESS,
    "episode_returns": episode_returns,
    "episode_lengths": episode_lengths,
    "successes": successes,
    "crashes": crashes,
    "timeouts": timeouts,
    "invalid_outputs": invalid_outputs,
  }

def get_test_score(tree, seeds=VALIDATION_SEEDS, episode_duration=1000):
    episode_returns = []
    episode_lengths = []
    successes = 0
    crashes = 0
    timeouts = 0
    invalid_outputs = 0

    for seed in seeds:
      observation = env.reset(seed=seed)[0]
      episode_return = 0.0
      final_reward = 0.0
      terminated = False
      truncated = False
      step_count = 0

      for step_count in range(1, episode_duration + 1):
        input_sample = torch.from_numpy(observation.reshape((1,-1))).float()
        output = tree.get_output_pt(input_sample)
        if not torch.isfinite(output).all():
          invalid_outputs += 1
          episode_return = INVALID_FITNESS
          break

        action = torch.argmax(output)
        observation, reward, terminated, truncated, info = env.step(action.item())
        final_reward = reward
        episode_return += reward
        if (terminated or truncated):
            break

      episode_returns.append(episode_return)
      episode_lengths.append(step_count)
      landed = terminated and final_reward >= 100 and observation[6] == 1.0 and observation[7] == 1.0
      successes += int(landed)
      crashes += int(terminated and not landed)
      timeouts += int(truncated or (not terminated and step_count >= episode_duration))

    return _make_diagnostics(
      episode_returns, episode_lengths, successes, crashes, timeouts, invalid_outputs=invalid_outputs
    )

best = evo.best_of_gens[-1]

print(best.get_readable_repr())

scores = get_test_score(best)
print(scores)

best_idx = int(np.argmax(scores["episode_returns"]))
best_seed = VALIDATION_SEEDS[best_idx]
print("Best seed:", best_seed)

## Make an animation
Here the best evolved individual is selected and one episode is rendered. Make sure to save your lunar landers over time to track progress and make comparisons.

In [ ]:
frames = []

def render_episode(tree, seed, episode_duration=1000):
    frames = []
    obs = env.reset(seed=seed)[0]

    for _ in range(episode_duration):
        frames.append(env.render())

        x = torch.from_numpy(obs.reshape((1, -1))).float()
        action = torch.argmax(tree.get_output_pt(x)).item()

        obs, reward, terminated, truncated, info = env.step(action)

        if terminated or truncated:
            frames.append(env.render())
            break

    return frames

# gist to save gif from https://gist.github.com/botforge/64cbb71780e6208172bbf03cd9293553
def save_frames_as_gif(frames, path='./', filename='evolved_lander.gif'):
  plt.figure(figsize=(frames[0].shape[1] / 72.0, frames[0].shape[0] / 72.0), dpi=72)
  patch = plt.imshow(frames[0])
  plt.axis('off')
  def animate(i):
      patch.set_data(frames[i])
  anim = animation.FuncAnimation(plt.gcf(), animate, frames = len(frames), interval=50)
  anim.save(path + filename, writer='imagemagick', fps=60)

frames = render_episode(best, best_seed)
env.close()
save_frames_as_gif(frames)

## Play animation

<img src="evolved_lander.gif" width="750">

## Optimisation
The coefficients in the multi-tree aren't optimised. Here Q-learning (taken from https://pytorch.org/tutorials/intermediate/reinforcement_q_learning.html) is used to optimise the weights further. Incorporate coefficient optimisation in training your agent(s). Coefficient Optimisation can be expensive. Think about how often you want to optimise, when, which individuals etc.

In [ ]:
batch_size = 128
GAMMA = 0.99

constants = best.get_subtrees_consts()

if len(constants)>0:
  optimizer = optim.AdamW(constants, lr=1e-3, amsgrad=True)

for _ in range(500):

  if len(constants)>0 and len(evo.memory)>batch_size:
    target_tree = copy.deepcopy(best)

    transitions = evo.memory.sample(batch_size)
    batch = Transition(*zip(*transitions))
    
    non_final_mask = torch.tensor(tuple(map(lambda s: s is not None,
                                        batch.next_state)), dtype=torch.bool)

    non_final_next_states = torch.cat([s for s in batch.next_state
                                               if s is not None])
    state_batch = torch.cat(batch.state)
    action_batch = torch.cat(batch.action)
    reward_batch = torch.cat(batch.reward)

    state_action_values = best.get_output_pt(state_batch).gather(1, action_batch)
    next_state_values = torch.zeros(batch_size, dtype=torch.float)
    with torch.no_grad():
      next_state_values[non_final_mask] = target_tree.get_output_pt(non_final_next_states).max(1)[0].float()

    expected_state_action_values = (next_state_values * GAMMA) + reward_batch
    
    criterion = nn.SmoothL1Loss()
    loss = criterion(state_action_values, expected_state_action_values.unsqueeze(1))
   
    # Optimize the model
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_value_(constants, 100)
    optimizer.step()

print(best.get_readable_repr())

scores = get_test_score(best)
print(scores)

best_idx = int(np.argmax(scores["episode_returns"]))
best_seed = VALIDATION_SEEDS[best_idx]
print("Best seed:", best_seed)

In [ ]:
frames = []
frames = render_episode(best, best_seed)
env.close()
save_frames_as_gif(frames, filename='evolved_lander_RL.gif')

<img src="evolved_lander_RL.gif" width="750">

## Function Sets

In [ ]:
import json
import os
import hashlib
import scipy.stats

def make_cache_key(internal_nodes, leaf_nodes, n_runs, pop_size, max_gens, seed, max_tree_size):
    key_dict = {
        "internal_nodes": sorted([type(n).__name__ for n in internal_nodes]),
        "leaf_nodes": sorted([type(n).__name__ for n in leaf_nodes]),
        "n_runs": n_runs,
        "max_tree_size": max_tree_size,
        "pop_size": pop_size,
        "max_gens": max_gens,
        "seed": seed,
    }
    key_str = json.dumps(key_dict, sort_keys=True)
    return hashlib.md5(key_str.encode()).hexdigest()

In [ ]:
def run_function_set_experiment(
    fitness_function,
    leaf_nodes,
    function_sets: dict,
    n_runs: int = 10,
    n_jobs: int = 12,
    pop_size: int = 64,
    max_gens: int = 20,
    seed: int = 42,
    max_tree_size=32,
    cache_file: str = "exp_results_improved_fitness_func.json",
):
    """
    Runs each function set n_runs times and returns mean fitness per set.
    Use results to classify nodes into function groups:
      - Two sets that perform the same -> nodes in the same group
      - A set that performs worse when a node is removed -> that node is in its own group
    Significance is tested with a two-tailed t-test (p < 0.05), matching the paper.
    """
    # load existing cache
    if os.path.exists(cache_file):
        with open(cache_file, 'r') as f:
            cache = json.load(f)
        print(f"Loaded cache with {len(cache)} existing entries")
    else:
        cache = {}

    results = {}

    for set_name, internal_nodes in function_sets.items():
        cache_key = make_cache_key(internal_nodes, leaf_nodes, n_runs, pop_size, max_gens, seed, max_tree_size)

        if cache_key in cache:
            print(f"Skipping {set_name} (cached)")
            results[set_name] = cache[cache_key]["scores"]
            continue

        print(f"Starting {set_name}...")
        scores = []
        for run in range(n_runs):
            print(f"Run number: {run}")
            evo = Evolution(
                fitness_function,
                internal_nodes,
                leaf_nodes,
                n_trees=4,
                pop_size=pop_size,
                max_tree_size=max_tree_size,
                max_gens=max_gens,
                n_jobs=n_jobs,
                seed=seed + run,
                verbose=False,
            )
            evo.evolve()
            scores.append(evo.best_of_gens[-1].fitness)

        results[set_name] = scores
        mean = np.mean(scores)
        print(f"{set_name}: mean={mean:.2f}, std={np.std(scores):.2f}  {[type(n).__name__ for n in internal_nodes]}")

        cache[cache_key] = {
            "internal_nodes": [type(n).__name__ for n in internal_nodes],
            "leaf_nodes": [type(n).__name__ for n in leaf_nodes],
            "n_runs": n_runs,
            "max_tree_size": max_tree_size,
            "pop_size": pop_size,
            "max_gens": max_gens,
            "seed": seed,
            "scores": scores,
            "mean": mean,
        }
        with open(cache_file, 'w') as f:
            json.dump(cache, f, indent=2)
        print(f"Saved to cache")

    # pairwise significance tests
    print("\n--- Pairwise t-tests (p < 0.05 = significantly different) ---")
    set_names = list(results.keys())
    for i in range(len(set_names)):
        for j in range(i + 1, len(set_names)):
            a, b = set_names[i], set_names[j]
            stat, p = scipy.stats.ttest_ind(results[a], results[b])
            sig = "DIFFERENT" if p < 0.05 else "same"
            print(f"  {a} vs {b}: p={p:.3f} -> {sig}")

    return results

In [ ]:
from genepro.node_impl import Plus, Minus, Times, Div, Square, Cube, Sqrt, Log, Exp, Sin, Cos, Max, Min, Abs, Tanh, IfGt

# Hypothesized function groups:
#   G1 - plus:          {Plus}
#   G2 - minus:         {Minus}
#   G3 - multiplicative:{Times, Div}
#   G4 - power/root:    {Sqrt, Square, Cube}
#   G5 - exp/log:       {Log, Exp}
#   G6 - periodic/bound:{Sin, Cos, Tanh}
#   G7 - selection:     {Max, Min, IfGt}
#   G8 - absolute:      {Abs}
#
# ref = one from each group: Plus, Minus, Times, Sqrt, Log, Sin, Max, Abs

function_sets = {
    "ref":                  [Plus(), Minus(), Times(), Sqrt(), Log(), Sin(), Max(), Abs()],

    # phase 1: within-group equivalence
    # For each group: swap individual members, pairs, and the full group.
    # Same as ref = in the same group. Better/worse = different groups.

    # G3
    "g3_div":               [Plus(), Minus(), Div(),   Sqrt(), Log(), Sin(), Max(), Abs()],  # swap Times→Div
    "g3_both":              [Plus(), Minus(), Times(), Div(),  Sqrt(), Log(), Sin(), Max(), Abs()],  # keep both

    # G4
    "g4_square":            [Plus(), Minus(), Times(), Square(),         Log(), Sin(), Max(), Abs()],  # swap Sqrt→Square
    "g4_cube":              [Plus(), Minus(), Times(), Cube(),           Log(), Sin(), Max(), Abs()],  # swap Sqrt→Cube
    "g4_sqrt_square":       [Plus(), Minus(), Times(), Sqrt(), Square(), Log(), Sin(), Max(), Abs()],  # keep Sqrt, add Square
    "g4_sqrt_cube":         [Plus(), Minus(), Times(), Sqrt(), Cube(),   Log(), Sin(), Max(), Abs()],  # keep Sqrt, add Cube
    "g4_square_cube":       [Plus(), Minus(), Times(), Square(), Cube(), Log(), Sin(), Max(), Abs()],  # swap Sqrt→Square+Cube
    "g4_all":               [Plus(), Minus(), Times(), Sqrt(), Square(), Cube(), Log(), Sin(), Max(), Abs()],  # all three

    # G5
    "g5_exp":               [Plus(), Minus(), Times(), Sqrt(), Exp(),         Sin(), Max(), Abs()],  # swap Log→Exp
    "g5_both":              [Plus(), Minus(), Times(), Sqrt(), Log(), Exp(),   Sin(), Max(), Abs()],  # keep both

    # G6
    "g6_cos":               [Plus(), Minus(), Times(), Sqrt(), Log(), Cos(),                  Max(), Abs()],  # swap Sin→Cos
    "g6_tanh":              [Plus(), Minus(), Times(), Sqrt(), Log(), Tanh(),                 Max(), Abs()],  # swap Sin→Tanh
    "g6_sin_cos":           [Plus(), Minus(), Times(), Sqrt(), Log(), Sin(), Cos(),            Max(), Abs()],  # keep Sin, add Cos
    "g6_sin_tanh":          [Plus(), Minus(), Times(), Sqrt(), Log(), Sin(), Tanh(),           Max(), Abs()],  # keep Sin, add Tanh
    "g6_cos_tanh":          [Plus(), Minus(), Times(), Sqrt(), Log(), Cos(), Tanh(),           Max(), Abs()],  # swap Sin→Cos+Tanh
    "g6_all":               [Plus(), Minus(), Times(), Sqrt(), Log(), Sin(), Cos(), Tanh(),    Max(), Abs()],  # all three

    # G7
    "g7_min":               [Plus(), Minus(), Times(), Sqrt(), Log(), Sin(), Min(),            Abs()],  # swap Max→Min
    "g7_ifgt":              [Plus(), Minus(), Times(), Sqrt(), Log(), Sin(), IfGt(),           Abs()],  # swap Max→IfGt
    "g7_max_min":           [Plus(), Minus(), Times(), Sqrt(), Log(), Sin(), Max(), Min(),     Abs()],  # keep Max, add Min
    "g7_max_ifgt":          [Plus(), Minus(), Times(), Sqrt(), Log(), Sin(), Max(), IfGt(),    Abs()],  # keep Max, add IfGt
    "g7_min_ifgt":          [Plus(), Minus(), Times(), Sqrt(), Log(), Sin(), Min(), IfGt(),    Abs()],  # swap Max→Min+IfGt
    "g7_all":               [Plus(), Minus(), Times(), Sqrt(), Log(), Sin(), Max(), Min(), IfGt(), Abs()],  # all three

    # phase 2: group necessity (remove one group entirely)
    # significantly worse than ref -> group is needed.
    # same or better -> group is redundant or harmful.
    "no_plus":              [Minus(), Times(), Sqrt(), Log(), Sin(), Max(), Abs()],
    "no_minus":             [Plus(),  Times(), Sqrt(), Log(), Sin(), Max(), Abs()],
    "no_mult":              [Plus(), Minus(), Sqrt(), Log(), Sin(), Max(), Abs()],
    "no_power":             [Plus(), Minus(), Times(), Log(), Sin(), Max(), Abs()],
    "no_explog":            [Plus(), Minus(), Times(), Sqrt(), Sin(), Max(), Abs()],
    "no_trig":              [Plus(), Minus(), Times(), Sqrt(), Log(), Max(), Abs()],
    "no_selection":         [Plus(), Minus(), Times(), Sqrt(), Log(), Sin(), Abs()],
    "no_abs":               [Plus(), Minus(), Times(), Sqrt(), Log(), Sin(), Max()],
}

results = run_function_set_experiment(
    fitness_function=fitness_function_pt3,
    leaf_nodes=leaf_nodes,
    function_sets=function_sets,
    n_runs=10,
    max_tree_size=63,
    pop_size=32,
    max_gens=20,
    seed=42,
)